# Notebook zum QR-Algorithmus

### Vorbereitungen

Wir benötigten in diesem Notebook die folgenden Module:

In [ ]:
import numpy as np
import scipy.linalg as spla  # für Matrixzerlegungen und co
import numpy.random as rnd   # für alles, was mit Zufallszahlen zu tun hat

Die Prozeduren, die wir in diesem Notebook betrachten, liefern verschiedene Vektoren und Matrizen als Ergebnis. Um diese schöner darstellen zu können, eignen sich die folgenden beiden Prozeduren. Was diese Prozeduren genau tun müssen Sie sich nicht anschauen.

In [ ]:
def printvector(v):
    if v.dtype == "int":
        print(''.join([' {:4}'.format(item) for item in v])+"\n")
    elif v.dtype == "complex128":
        print(''.join([' {:16.3f}'.format(item) for item in v])+"\n")
    else:
        print(''.join([' {:7.3f}'.format(item) for item in v])+"\n")

In [ ]:
def printmatrix(A):
    if A.dtype == "int":
        print('\n'.join([''.join(['  {:4}'.format(item) for item in row]) for row in A])+"\n")
    elif A.dtype == "complex128":
        print('\n'.join([''.join(['  {:16.3f}'.format(item) for item in row]) for row in A])+"\n")   
    else:
        print('\n'.join([''.join(['  {:7.3f}'.format(item) for item in row]) for row in A])+"\n")       

### Problemstellung & Modellmatrizen
In diesem Notebook wollen wir Eigenwerte von Matrizen mit verschiedenen Varianten des QR-Algorithmus approximieren. Zunächst konstruieren wir uns dazu ein paar Modellmatrizen, von denen wir die Eigenwerte kennen und mit denen wir die Algorithmen testen können.

Zur Konstruktion der Matrizen starten wir zunächst mit einer Diagonalmatrix oder einer Blockdiagonalmatrix $D$, und wenden dann eine beliebig ausgewählte Ähnlichkeitstransformation an, d.h. berechnen $A=S^{-1} D S$ für eine invertierbare Matrix $S$. Die Eigenwerte von $A$ entsprechen dann denen von $D$. Der Matrix $A$ selbst sieht man aber die Eigenwerte nicht direkt an.

Konkret verwenden wir
$$
D_1 = \begin{pmatrix} 2 \\ & 1 \\ && 5 \\ &&& -4 \\ &&&& \frac12 \end{pmatrix}, \qquad
D_2= \begin{pmatrix} 2+2\mathrm{i} \\ & 1 \\ && 5 \\ &&& -4+1\mathrm{i} \\ &&&& \frac12-3\mathrm{i} \end{pmatrix}, \qquad
D_3 = \begin{pmatrix} 2 \\ & 1 \\ && 5 \\ &&& 5 \\ &&&& \frac12 \end{pmatrix}, \qquad
D_4 = \begin{pmatrix} 2 \\ & 1 \\ && 5 \\ &&& -5 \\ &&&& \frac12 \end{pmatrix}, \qquad
$$
sowie
$$
D_5 = \begin{pmatrix} 2+2\mathrm{i} \\ & 1 \\ && 1 & -1 \\ && 1 & 1 \\ &&&& \frac12-3\mathrm{i} \end{pmatrix} 
\qquad \text{und} \qquad
D_6 = \begin{pmatrix} 2 \\ & 1 \\ && 1 & -1 \\ && 1 & 1 \\ &&&& \frac12 \end{pmatrix},
$$
und definieren dann $A_i = S^{-1} D_i S$ für $i=1,\ldots,6$ mit der invertierbaren Matrix
$$ 
S = \begin{pmatrix} 
     2 & -1 &  1 &  0 & -1 \\
    -1 &  1 &  2 &  2 &  2 \\
    -1 &  0 &  2 & -1 &  1 \\
    -1 &  2 &  2 &  2 &  1 \\
     2 & -1 &  2 &  0 & -1 
\end{pmatrix}.
$$ 

In [ ]:
# (Block-)Diagonalmatrizen
A1 = np.diag(np.array([2, 1, 5, -4, 1/2]))
A2 = np.diag(np.array([2+2j, 1, 5, -4+1j, 1/2-3j]))
A3 = np.diag(np.array([2, 1, 5,  5, 1/2]))
A4 = np.diag(np.array([2, 1, 5, -5, 1/2]))
A5 = np.array([[2+2j,0,0,0,0],[0,1,0,0,0],[0,0,1,-1,0],[0,0,1,1,0],[0,0,0,0,1/2-3j]])
A6 = np.array([[2,0,0,0,0],[0,1,0,0,0],[0,0,1,-1,0],[0,0,1,1,0],[0,0,0,0,1/2]])

# Ähnlichkeitstransformation
S = np.array([
    [2, -1, 1, 0, -1],
    [-1, 1, 2, 2, 2],
    [-1, 0, 2, -1, 1],
    [-1, 2, 2, 2, 1],
    [2, -1, 2, 0, -1]
])
S_inv = spla.inv(S)

A1 = S_inv @ A1 @ S
A2 = S_inv @ A2 @ S
A3 = S_inv @ A3 @ S
A4 = S_inv @ A4 @ S
A5 = S_inv @ A5 @ S
A6 = S_inv @ A6 @ S

Für die Matrix $A_1$ gilt dann zum Beispiel:

In [ ]:
print('A_1 =')
printmatrix(A1)

**(a) Nennen Sie kurz die "Herausforderungen" im Bezug auf die Eigenwerte von $D_3, \ldots, D_6$.**

## 1.) Naiver QR-Algorithmus

**(b) Implementieren Sie eine Prozedur `qr_alg_naiv`, die den QR-Algorithmus in der naiven Version auf eine Matrix anwendet. Die Anzahl der Iterationen `kMax` soll dabei als Eingabeparameter übergeben werden.**

Berechnen Sie die QR-Zerlegungen mithilfe der in `scipy` enthalten Prozedur (also mit `spla.qr(...)`).

**(c) Wenden Sie die Prozedur zunächst auf die Matrizen $A_1$ und $A_2$ an. Wie viele Iterationen `kMax` brauchen Sie für gute Ergebnisse?**

**(d) Wenden Sie die Prozedur nun jeweils mit `kMax = 50` auf die Matrizen $A_3,...,A_6$ an. Beobachten und erklären Sie die Ergebnisse.**

## 2.) Transformation auf Hessenberg-Form

Als ersten Schritt hin zu einer effizienteren Implementierung wollen wir die Matrizen durch eine Ähnlichkeitstransformation auf Hessenberg-Form bringen.

**(e) Schreiben Sie eine Prozedur `hess`, die eine Matrix durch eine unitäre Ähnlichkeitstransformation auf Hessenberg-Form bringt (siehe Beweis von Satz 6.22).**

Hinweise:
* Berücksichtigen Sie, wie die richtige Householder-Transformation aussieht, die einen Vektor mit **komplexen** Einträgen auf ein Vielfaches des ersten Einheitsvektors spiegelt (siehe Abschnitt 6.8.1 im Skript).
* Die verwendeten Householder-Transformationen brauchen Sie nicht speichern, denn wir sind nur an den Eigenwerten von Matrizen interessiert, und die Hessenberg-Matrix, die Ihre Prozedur am Ende liefert, ist ja ähnlich zur Ausgangsmatrix.
* Bei Vektoren (genauer: eindimensionale `ndarray`) unterscheidet Numpy nicht zwischen Zeilen- und Spaltenvektoren, sondern interpretiert sie (zum Beispiel bei der Matrixmultilplikation) so wie es Sinn macht.
* Die komplex konjugierte Variante eines Vektors/arrays $v$ erhalten Sie über `v.conj()`.
* Das dyadische Produkt $vw^\top$ zweier Vektoren $v,w\in\mathbb{C}^n$ erhalten Sie über `np.outer(v,w)`, und dementsprechend das Produkt $vw^H$ über `np.outer(v,w.conj())`.

**(f) Wenden Sie Ihre Prozedur zum Test auf die Matrizen $A_1$ und $A_2$ an. Übergeben Sie dabei nur eine Kopie der Matrizen an die Prozedur `hess`, damit die Ausgangsmatrizen unverändert bleiben. Überprüfen Sie: Haben die Ergebnismatrizen Hessenberg-Struktur? Haben Sie weiterhin die selben Eigenwerte wie die Ausgangsmatrizen (über `spla.eigvals` überprüfbar)?**

## 3.) QR-Algorithmus mit Shift für Hessenberg Matrizen

Bevor wir uns um den QR-Algorithmus mit Shift kümmern, bringen wir zunächst alle Modellmatrizen in Hessenberg-Form:

In [ ]:
A1_hess = hess(A1.copy())
A2_hess = hess(A2.copy())
A3_hess = hess(A3.copy())
A4_hess = hess(A4.copy())
A5_hess = hess(A5.copy())
A6_hess = hess(A6.copy())

Nun wollen wir den QR-Algorithmus mit Shift für eine Hessenberg-Matrix $H$ implementieren. Dabei verwenden wird immer das letzte Element $\mu = h_{n,n}$ von $H$ als Shift. Wir iterieren so lange, bis Deflation auftritt, in dem Sinne, dass
$$
|h_{n,n-1}| \leq \texttt{eps} \left( |h_{n-1,n-1,}| + |h_{n,n}| \right)
$$
mit der Maschinengenauigkeit $\texttt{eps}$ gilt.

**(g) Implementieren Sie den eben beschriebenen QR-Algorithmus mit Shift. Sobald Deflation auftritt, soll die Prozedur beendet werden und dabei den isolierten Eigenwert sowie die Restmatrix ausgeben.**

Hinweise:
* Verwenden Sie wieder einen Parameter `kMax`, um die maximale Iterationszahl festzulegen, falls es nicht vorzeitig zu Deflation kommen sollte. In diesem Fall soll nur die letzte Iterierte ausgegeben werden.
* Die Maschinengenauigkeit (für 64Bit-Gleitkommazahlen) erhalten Sie über `np.finfo(np.float64).eps`.
* Stellen Sie sicher, dass Ihre Prozedur auch für $1\times1$-Matrizen ein sinnvolles Ergebnis liefert.

Testen Sie Ihre Prozedur mit den Matrizen `A1_hess` und `A2_hess`. Lassen Sie Python auch die Eigenwerte der Restmatrix berechnen und überprüfen Sie, ob die Ergebnisse Sinn ergeben.

**(h) Erweitern Sie Ihre Prozedur aus Teil (g) folgendermaßen: Sobald Deflation auftritt, rufen Sie die Prozedur rekursiv auf, um den QR-Algorithmus mit der Restmatrix erneut zu starten. Speichern Sie dann den isolierten Eigenwert sowie die Eigenwerte der Restmatrix in einem Vektor, den Sie am Ende zurückgeben. Achten Sie hier besonders darauf, dass Ihre Prozedur für $1\times1$-Matrizen sinnvoll agiert.**

**(i) Sofern kein Abbruch wegen Erreichen der maximalen Iterationszahl erfolgt, sollte Ihre Prozedur aus Teil (h) einen Vektor mit allen Eigenwerten der eingegebenen Matrix $H$ berechnen. Überprüfen Sie dies zunächst anhand der Matrizen `A1_hess` und `A2_hess`. Wenn für diese Matrizen alles funktioniert, testen Sie auch die Matrizen `A3_hess`,...,`A6_hess`. Mit welchen Matrizen kommt die Prozedur (nicht) klar? Haben Sie eine Idee, warum? Beobachten Sie außerdem auch die Anzahl an Iterationen, die insgesamt durchgeführt wird, insbesondere im Vergleich zu den Aufgabenteilen (c) und (d).**

Natürlich können wir unsere Prozedur jetzt auch auf andere Matrizen als die Modellmatrizen anwenden, und so die Eigenwerte (quasi) beliebiger Matrizen berechnen. Hier zum Beispiel für eine zufällige Matrix mit komplexen Einträgen: 

In [ ]:
n = 20
A = rnd.rand(n,n) + 1j*rnd.rand(n,n)
A_hess = hess(A)

print('Eigenwerte laut Python:')
printvector(spla.eigvals(A_hess))

print('Eigenwerte selbst berechnet:')
printvector(qr_alg_shift(A_hess,50))